# 5.2 — Phasor GAN from First Principles

This notebook builds a **Generative Adversarial Network (GAN)** from scratch using only `PhasorCircuit` and the `AnalyticEngine` backend. No high-level model wrappers — every circuit is constructed manually so the architecture is fully transparent.

### Architecture

| Component | Circuit | Readout |
|---|---|---|
| **Generator** | $z \to$ `shift` $\to$ `dft` $\times L$ | $\sin(\text{phases})$ |
| **Discriminator** | $\arcsin(x) \to$ `shift` $\to$ `mix` $\to$ `dft` $\times L$ | $(\sin(\theta_0)+1)/2$ |

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import math
import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt

import PhasorFlow as pf
from PhasorFlow.circuit import PhasorCircuit
from PhasorFlow.engine.analytic import AnalyticEngine

np.random.seed(42)
torch.manual_seed(42)

# Shared backend
backend = AnalyticEngine()

## 1. Synthetic Timeseries Data

Composite waveform: $\sin(t) + 0.5\cos(3t) + 0.3\sin(5t)$, windowed to length $T=8$.

In [ ]:
T = 8             # Sequence length = number of phasor threads
N_SAMPLES = 100

t_axis = np.linspace(0, 8 * np.pi, N_SAMPLES + T)
signal = np.sin(t_axis) + 0.5 * np.cos(3 * t_axis) + 0.3 * np.sin(5 * t_axis)
signal = signal / np.max(np.abs(signal))           # normalize to [-1, 1]
signal += np.random.normal(0, 0.02, len(signal))
signal = np.clip(signal, -0.99, 0.99)

X_real = torch.tensor(
    [signal[i:i+T] for i in range(N_SAMPLES)], dtype=torch.float32
)
print(f'Real data: {X_real.shape}, range [{X_real.min():.3f}, {X_real.max():.3f}]')

plt.figure(figsize=(10, 3))
for i in range(6):
    plt.plot(X_real[i].numpy(), alpha=0.6)
plt.title('Real Timeseries Samples'); plt.xlabel('Step'); plt.ylabel('Amplitude')
plt.tight_layout(); plt.show()

## 2. Generator Circuit (First Principles)

The generator takes random noise $z \in [-\pi, \pi]^T$ and passes it through a phasor circuit with trainable shift layers interleaved with DFT gates. The output phases are mapped to signal values via $\sin(\cdot)$.

```
Thread 0: ──[Shift z₀]──[Shift w₀]──[DFT]──[Shift w₈]──[DFT]── → sin(θ₀)
Thread 1: ──[Shift z₁]──[Shift w₁]──[DFT]──[Shift w₉]──[DFT]── → sin(θ₁)
  ...         ...          ...                 ...                    ...
Thread 7: ──[Shift z₇]──[Shift w₇]──[DFT]──[Shift w₁₅]─[DFT]── → sin(θ₇)
```

In [ ]:
G_LAYERS = 2
G_PARAMS = G_LAYERS * T  # 2 layers × 8 threads = 16 parameters

# Trainable generator weights
g_weights = torch.empty(G_PARAMS).uniform_(-math.pi / 4, math.pi / 4).requires_grad_(True)

def build_generator_circuit(z, weights):
    """
    Build a generator PhasorCircuit from scratch.
    
    z:       latent noise vector, shape (T,)
    weights: trainable shifts, shape (G_LAYERS * T,)
    """
    pc = PhasorCircuit(T, name='Generator')
    
    # Step 1: Encode latent noise as initial phases
    for i in range(T):
        pc.shift(i, z[i])
    
    # Step 2: Variational layers (Shift → DFT)
    for layer in range(G_LAYERS):
        offset = layer * T
        for i in range(T):
            pc.shift(i, weights[offset + i])   # trainable phase shift
        pc.dft()                                # global DFT mixing
    
    return pc

def generator_forward(z, weights):
    """Run generator circuit → sin(phases) → fake sample."""
    pc = build_generator_circuit(z, weights)
    result = backend.run(pc)
    return torch.sin(result['phases'])  # map phases to [-1, 1]

def generate_fake_batch(weights, batch_size):
    """Generate a batch of fake samples from random noise."""
    z_batch = torch.empty(batch_size, T).uniform_(-math.pi, math.pi)
    return torch.stack([generator_forward(z_batch[i], weights) for i in range(batch_size)])

# Preview
z_test = torch.randn(T)
pc_gen = build_generator_circuit(z_test, g_weights)
print(f'Generator circuit: {pc_gen.num_threads} threads, {len(pc_gen.operations)} operations')
print(f'Generator params:  {G_PARAMS}')
fake_sample = generator_forward(z_test, g_weights)
print(f'Untrained output:  {fake_sample.detach().numpy()}')

## 3. Discriminator Circuit (First Principles)

The discriminator takes a timeseries sample, encodes it into phases via $\arcsin(x)$, then processes it through a phasor circuit with trainable shifts, mix coupling, and DFT. Thread 0's output phase is mapped to a probability.

```
Thread 0: ──[Shift arcsin(x₀)]──[Shift w₀]──[Mix 0,1]──[DFT]──...── → (sin(θ₀)+1)/2
Thread 1: ──[Shift arcsin(x₁)]──[Shift w₁]──[Mix 0,1]──[DFT]──...──
  ...         ...                  ...          ...                      ...
Thread 7: ──[Shift arcsin(x₇)]──[Shift w₇]──[Mix 6,7]──[DFT]──...──
```

In [ ]:
D_LAYERS = 2
D_PARAMS = D_LAYERS * T  # 2 layers × 8 threads = 16 parameters

# Trainable discriminator weights
d_weights = torch.empty(D_PARAMS).uniform_(-math.pi / 4, math.pi / 4).requires_grad_(True)

def build_discriminator_circuit(x_phases, weights):
    """
    Build a discriminator PhasorCircuit from scratch.
    
    x_phases: phase-encoded input, shape (T,)
    weights:  trainable shifts, shape (D_LAYERS * T,)
    """
    pc = PhasorCircuit(T, name='Discriminator')
    
    # Step 1: Encode input signal as phases
    for i in range(T):
        pc.shift(i, x_phases[i])
    
    # Step 2: Variational layers (Shift → Mix pairs → DFT)
    for layer in range(D_LAYERS):
        offset = layer * T
        for i in range(T):
            pc.shift(i, weights[offset + i])   # trainable phase shift
        for i in range(0, T - 1, 2):            # alternating mix pairs
            pc.mix(i, i + 1)
        pc.dft()                                # global DFT mixing
    
    return pc

def discriminator_forward(x, weights):
    """Run discriminator: encode → circuit → P(real)."""
    x_clamped = torch.clamp(x, -0.999, 0.999)
    x_phases = torch.asin(x_clamped)            # phase encoding
    
    pc = build_discriminator_circuit(x_phases, weights)
    result = backend.run(pc)
    
    # Readout: sine envelope on thread 0 → probability in [0, 1]
    return (torch.sin(result['phases'][0]) + 1.0) / 2.0

def discriminator_batch(X, weights):
    """Classify a batch of samples."""
    return torch.stack([discriminator_forward(X[i], weights) for i in range(X.shape[0])])

# Preview
pc_disc = build_discriminator_circuit(torch.zeros(T), d_weights)
print(f'Discriminator circuit: {pc_disc.num_threads} threads, {len(pc_disc.operations)} operations')
print(f'Discriminator params:  {D_PARAMS}')
p_real = discriminator_forward(X_real[0], d_weights)
print(f'Untrained P(real) for sample 0: {p_real.item():.4f}')

## 4. Adversarial Training Loop

Standard minimax GAN training with separate Adam optimizers:

$$\min_G \max_D \;\; \mathbb{E}[\log D(x)] + \mathbb{E}[\log(1 - D(G(z)))]$$

In [ ]:
EPOCHS = 60
BATCH_SIZE = 10
LR = 0.01

opt_g = optim.Adam([g_weights], lr=LR)
opt_d = optim.Adam([d_weights], lr=LR)

history = {'d_loss': [], 'g_loss': [], 'd_real': [], 'd_fake': []}

print(f'Training Phasor GAN from First Principles')
print(f'  T={T}, G_layers={G_LAYERS}, D_layers={D_LAYERS}')
print(f'  G_params={G_PARAMS}, D_params={D_PARAMS}')
print()

for epoch in range(EPOCHS):
    # --- Sample mini-batch ---
    idx = torch.randint(0, N_SAMPLES, (BATCH_SIZE,))
    real_batch = X_real[idx]
    fake_batch = generate_fake_batch(g_weights, BATCH_SIZE)
    
    # === Train Discriminator ===
    opt_d.zero_grad()
    
    d_real_out = discriminator_batch(real_batch, d_weights)
    d_loss_real = -torch.mean(torch.log(d_real_out + 1e-8))
    
    d_fake_out = discriminator_batch(fake_batch.detach(), d_weights)
    d_loss_fake = -torch.mean(torch.log(1.0 - d_fake_out + 1e-8))
    
    d_loss = d_loss_real + d_loss_fake
    d_loss.backward()
    opt_d.step()
    
    # === Train Generator ===
    opt_g.zero_grad()
    
    fake_batch_g = generate_fake_batch(g_weights, BATCH_SIZE)
    d_fake_for_g = discriminator_batch(fake_batch_g, d_weights)
    g_loss = -torch.mean(torch.log(d_fake_for_g + 1e-8))
    g_loss.backward()
    opt_g.step()
    
    # Record
    history['d_loss'].append(d_loss.item())
    history['g_loss'].append(g_loss.item())
    history['d_real'].append(d_real_out.mean().item())
    history['d_fake'].append(d_fake_out.mean().item())
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | '
              f'D_loss: {d_loss.item():.4f} | G_loss: {g_loss.item():.4f} | '
              f'D(real): {d_real_out.mean().item():.3f} | '
              f'D(fake): {d_fake_out.mean().item():.3f}')

print('\nTraining complete!')

## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['d_loss'], label='D Loss', color='tab:blue')
axes[0].plot(history['g_loss'], label='G Loss', color='tab:red')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend()

axes[1].plot(history['d_real'], label='D(real)', color='tab:green')
axes[1].plot(history['d_fake'], label='D(fake)', color='tab:orange')
axes[1].axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)
axes[1].set_title('Discriminator Confidence')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Probability'); axes[1].legend()

plt.tight_layout(); plt.show()

## 6. Real vs Generated Samples

In [ ]:
with torch.no_grad():
    fake_samples = generate_fake_batch(g_weights, 8)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i in range(8):
    axes[0].plot(X_real[i].numpy(), alpha=0.6, color='tab:blue')
axes[0].set_title('Real Timeseries')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Amplitude')
axes[0].set_ylim(-1.5, 1.5)

for i in range(8):
    axes[1].plot(fake_samples[i].numpy(), alpha=0.6, color='tab:red')
axes[1].set_title('Generated Timeseries (First Principles Phasor GAN)')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Amplitude')
axes[1].set_ylim(-1.5, 1.5)

plt.tight_layout(); plt.show()

## 7. Circuit Introspection

Since we built the circuits manually, we can visualize them using PhasorFlow's drawing tools.

In [ ]:
# Visualize the Generator circuit
z_demo = torch.zeros(T)
pc_gen_demo = build_generator_circuit(z_demo, g_weights)
print('=== Generator Circuit ===')
pf.draw(pc_gen_demo, mode='text')

print()

# Visualize the Discriminator circuit
x_demo = torch.zeros(T)
pc_disc_demo = build_discriminator_circuit(x_demo, d_weights)
print('=== Discriminator Circuit ===')
pf.draw(pc_disc_demo, mode='text')

## 8. Summary

This notebook showed how the `PhasorGAN` model is built from first principles:

1. **Generator**: Random phases → `pc.shift()` (trainable) → `pc.dft()` → `sin(phases)` → fake data
2. **Discriminator**: `arcsin(x)` → `pc.shift()` (trainable) → `pc.mix()` → `pc.dft()` → `(sin(θ₀)+1)/2` → P(real)
3. **Training**: Standard minimax GAN with separate Adam optimizers

The high-level `PhasorGAN` class in `PhasorFlow.models.gan` wraps this exact logic into a convenient `fit()` / `generate()` API.

In [ ]:
print('Final D(real):', f"{history['d_real'][-1]:.4f}")
print('Final D(fake):', f"{history['d_fake'][-1]:.4f}")
print(f'Generator weights shape:     {g_weights.shape} ({g_weights.numel()} params)')
print(f'Discriminator weights shape: {d_weights.shape} ({d_weights.numel()} params)')